# train.ipynb — 최종 학습 (2차 평가 제출용)

**목적**: `05_tuning`에서 확정한 최종 설정으로 **train 전체 데이터(2022~2024)**를 재학습해 제출용 모델 파일을 만든다. 2차 평가 요건(`CLAUDE.md` 1절)에 따라 학습 코드(`train.ipynb`)와 추론 코드(`inference.ipynb`)를 분리한다 — 이 노트북은 오직 학습·모델 저장만 하고, 예측·제출 파일 생성은 `inference.ipynb`에서 한다.

**확정된 최종 설정** (`05_tuning.ipynb` 11~14절 / `reports/05_tuning.md` 근거):
- **CatBoost**: 통합모델(3개 그룹을 `group_id` 범주형 피처로 구분해 하나의 모델로 학습, 타깃은 이용률=발전량/설비용량), CatBoost 기본 하이퍼파라미터, 분위수 회귀(`Quantile:alpha=0.70`), 표본 가중치 `actual`(kWh) — 채점 대상이 실측값 기준(`actual >= capacity*0.10`)이고 FICR이 실측값 가중합이라는 `src/metric.py` 구조를 이용(05_tuning 9절, 3-fold CV +0.0305)
- **MLP**(05_tuning 14절, 신규): 같은 통합모델 구조에 `src/nn.py`의 `soft_metric_loss`(FICR 계단 함수를 시그모이드로 미분 가능하게 근사) — CatBoost의 quantile 손실이 채점 산식을 간접적으로만 우회하는 데 반해, MLP는 그 산식 자체를 직접 최적화한다. T_soft=0.003
- **CatBoost + MLP 블렌드**(14-8/14-9에서 seed 3개로 검증): 그룹별 블렌드 가중치 group_1=0.4 / group_2=0.5 / **group_3=1.0(MLP만, CatBoost 완전 배제)** — CV 0.6295(표준편차 0.0013), CatBoost 단독 대비 +0.0082(표준편차의 6.2배)로 검증된 진짜 개선
- 피처: `train_features_v2.parquet`의 50개 피처(`04_model_selection` 피처 선택 결과), 두 모델 공통 사용
- 앙상블: 각 모델을 seed 3개(42/7/2024)로 학습해 평균(`ensemble-final` 스킬 3절)

**입력**: `data/processed/train_features_v2.parquet`
**출력**: `models/catboost_seed{42,7,2024}.cbm`, `models/mlp_seed{42,7,2024}.pt`, `models/mlp_scaler.npz`(MLP 표준화 통계), `models/model_config.json`(재현에 필요한 모든 설정 — 블렌드 가중치 포함)

**early stopping 문제와 해결**: 전체 데이터로 재학습하면 검증셋이 없어 early stopping을 쓸 수 없다. 두 모델 모두 `ensemble-final` 스킬 4절 표준 관행대로, 05_tuning과 같은 3-fold CV 구조에서 이 최종 설정으로 다시 학습해 각 fold가 조기 종료로 멈춘 시점(CatBoost는 iteration, MLP는 epoch)을 확인하고, 그 평균의 1.05배를 전체 데이터 재학습의 고정값으로 사용한다.

## 0. 셋업

패키지를 불러오고 환경을 확인한다. `inference.ipynb`와 완전히 분리된 노트북이므로 필요한 import를 전부 이 노트북 안에서 다시 한다(`ensemble-final` 스킬 5절 — 두 노트북은 각각 단독으로 처음부터 끝까지 실행 가능해야 함).

In [1]:
import sys
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import catboost
import torch
from catboost import CatBoostRegressor

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "data").exists(), "REPO_ROOT를 찾지 못했습니다. 노트북 실행 위치를 확인하세요."

sys.path.insert(0, str(REPO_ROOT))
from src.metric import TARGET_COLS, CAPACITY_KWH
from src import nn as mlp_nn

PROCESSED_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

print("python:", sys.version)
print("python executable:", sys.executable)
print("REPO_ROOT:", REPO_ROOT)
print("pandas:", pd.__version__, "| numpy:", np.__version__, "| catboost:", catboost.__version__, "| torch:", torch.__version__)
print("OS:", platform.platform())

python: 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)]
python executable: d:\공모전\wind_forecast\venv\Scripts\python.exe
REPO_ROOT: d:\공모전\wind_forecast
pandas: 3.0.3 | numpy: 2.5.1 | catboost: 1.2.10 | torch: 2.13.0+cpu
OS: Windows-11-10.0.26200-SP0


## 1. 데이터 로드 (train 전체)

`04_model_selection`/`05_tuning`에서 쓴 v2 피처셋(50개)을 그대로 불러온다. train/test가 동일 로직으로 만들어졌는지는 `01_preprocessing`~`03_features`에서 이미 확인했으므로 여기서는 shape과 기간만 재확인한다.

In [2]:
train_features = pd.read_parquet(PROCESSED_DIR / "train_features_v2.parquet")

DROP_COLS = {"forecast_kst_dtm", "forecast_id", *TARGET_COLS}
FEATURE_COLS = [c for c in train_features.columns if c not in DROP_COLS]

print("train_features:", train_features.shape)
print("피처 개수:", len(FEATURE_COLS))
print("기간:", train_features["forecast_kst_dtm"].min(), "~", train_features["forecast_kst_dtm"].max())

train_features: (26304, 54)
피처 개수: 50
기간: 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00


## 2. 최종 설정 확정

`05_tuning.ipynb`에서 확정한 값을 그대로 하드코딩한다(재현성 — 다른 곳에서 값을 가져오지 않고 이 노트북만 봐도 전체 설정을 알 수 있게).

In [3]:
FINAL_HYPERPARAMS = {"learning_rate": 0.05}  # CatBoost 기본값. 05_tuning 8절: Optuna 튜닝 효과는 seed 표준편차보다 작아 채택하지 않음
QUANTILE_ALPHA = 0.70           # 05_tuning 9절: 3-fold CV +0.0305 개선(seed 표준편차의 47.6배로 안정적). peak(0.71)보다 안정적인 0.70 채택
USE_SAMPLE_WEIGHT = True        # 학습 표본 가중치 = actual(kWh) — FICR이 실측값 가중합이라는 산식 구조 반영
ENSEMBLE_SEEDS = [42, 7, 2024]  # 05_tuning에서 안정성 확인에 쓴 seed와 동일 — 재현성 유지. CatBoost/MLP 둘 다 이 seed로 앙상블
EARLY_STOP_ROUNDS = 100
ITERATIONS_CAP = 2000           # early stopping 탐색 시 상한(05_tuning과 동일)

# MLP 설정 (05_tuning 14절 확정값)
MLP_T_SOFT = 0.003        # 14-5 T_soft 스윕 최적값(0.003~0.02 구간이 평평해 민감하지 않음을 확인)
MLP_HIDDEN = (256, 256)
MLP_DROPOUT = 0.15
MLP_MAX_EPOCHS_CAP = 300  # early stopping 탐색 시 상한(05_tuning 14절과 동일)
MLP_PATIENCE = 30

# CatBoost+MLP 그룹별 블렌드 가중치 (05_tuning 14-8/14-9, seed 3개로 검증된 값)
# 값 = MLP 비중. group_3은 1.0 -> CatBoost를 아예 안 씀
BLEND_WEIGHTS = {"kpx_group_1": 0.4, "kpx_group_2": 0.5, "kpx_group_3": 1.0}

GROUP_ID_MAP = {"kpx_group_1": 0, "kpx_group_2": 1, "kpx_group_3": 2}
GROUP_ID_CATEGORIES = [0, 1, 2]

print("FINAL_HYPERPARAMS:", FINAL_HYPERPARAMS)
print("QUANTILE_ALPHA:", QUANTILE_ALPHA, "| USE_SAMPLE_WEIGHT:", USE_SAMPLE_WEIGHT)
print("ENSEMBLE_SEEDS:", ENSEMBLE_SEEDS)
print("MLP_T_SOFT:", MLP_T_SOFT, "| MLP_HIDDEN:", MLP_HIDDEN, "| MLP_DROPOUT:", MLP_DROPOUT)
print("BLEND_WEIGHTS:", BLEND_WEIGHTS)

FINAL_HYPERPARAMS: {'learning_rate': 0.05}
QUANTILE_ALPHA: 0.7 | USE_SAMPLE_WEIGHT: True
ENSEMBLE_SEEDS: [42, 7, 2024]
MLP_T_SOFT: 0.003 | MLP_HIDDEN: (256, 256) | MLP_DROPOUT: 0.15
BLEND_WEIGHTS: {'kpx_group_1': 0.4, 'kpx_group_2': 0.5, 'kpx_group_3': 1.0}


## 3. 최종 `iterations` 결정 — 3-fold CV로 early stopping 반복 횟수 확인

`ensemble-final` 스킬 4절: 전체 데이터로 재학습하면 검증셋이 사라져 early stopping을 못 쓴다 → **CV에서 관측된 최적 iteration의 평균(×1.05)으로 `iterations`를 고정**하는 게 표준 관행. `05_tuning`과 동일한 확장 윈도우 3-fold 구조로 이 최종 설정(τ=0.70, actual 가중, `FINAL_HYPERPARAMS`)을 다시 학습해서, 각 fold가 early stopping으로 멈춘 시점(`model.best_iteration_`)을 확인한다.

In [4]:
FOLDS = [
    {"name": "fold1", "train_start": "2022-01-01 01:00:00", "train_end": "2023-07-01 00:00:00"},
    {"name": "fold2", "train_start": "2022-01-01 01:00:00", "train_end": "2024-01-01 00:00:00"},
    {"name": "fold3", "train_start": "2022-01-01 01:00:00", "train_end": "2024-07-01 00:00:00"},
]


def make_fold_train(fold):
    dtm = train_features["forecast_kst_dtm"]
    train_start = pd.Timestamp(fold["train_start"])
    train_end = pd.Timestamp(fold["train_end"])
    return train_features[(dtm >= train_start) & (dtm <= train_end)].reset_index(drop=True)


def to_long(df):
    frames = []
    for g in TARGET_COLS:
        sub = df[df[g].notna()].copy()
        sub["group_id"] = GROUP_ID_MAP[g]
        sub["utilization"] = sub[g] / CAPACITY_KWH[g]
        sub["actual_kwh"] = sub[g]
        frames.append(sub[["forecast_kst_dtm", "group_id", "utilization", "actual_kwh"] + FEATURE_COLS])
    return pd.concat(frames, ignore_index=True)


features = FEATURE_COLS + ["group_id"]
best_iterations = []

for fold in FOLDS:
    fold_train = make_fold_train(fold)
    long_df = to_long(fold_train)
    long_df["group_id"] = pd.Categorical(long_df["group_id"], categories=GROUP_ID_CATEGORIES)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * 0.15)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={QUANTILE_ALPHA}",
        iterations=ITERATIONS_CAP,
        random_seed=42,
        verbose=False,
        **FINAL_HYPERPARAMS,
    )
    fit_kwargs = {"sample_weight": fit_rows["actual_kwh"].to_numpy()} if USE_SAMPLE_WEIGHT else {}
    model.fit(
        fit_rows[features], fit_rows["utilization"],
        eval_set=(early_rows[features], early_rows["utilization"]),
        cat_features=["group_id"], early_stopping_rounds=EARLY_STOP_ROUNDS, verbose=False,
        **fit_kwargs,
    )
    best_iterations.append(model.best_iteration_)
    print(f"{fold['name']}: best_iteration_={model.best_iteration_}")

mean_iter = float(np.mean(best_iterations))
FINAL_ITERATIONS = int(np.ceil(mean_iter * 1.05))
print(f"\n평균 best_iteration_: {mean_iter:.1f} -> 1.05배 적용 후 FINAL_ITERATIONS = {FINAL_ITERATIONS}")

fold1: best_iteration_=356
fold2: best_iteration_=518
fold3: best_iteration_=1170

평균 best_iteration_: 681.3 -> 1.05배 적용 후 FINAL_ITERATIONS = 716


## 3b. MLP 최종 epoch 수 결정 — 3-fold CV로 조기 종료 시점 확인

CatBoost와 같은 이유(전체 데이터 재학습 시 검증셋이 없어 early stopping 불가)로, MLP도 같은 3-fold CV 구조에서 미리 조기 종료 시점(`best_epoch`)을 확인해두고 평균×1.05를 전체 재학습의 고정 epoch 수로 쓴다(`ensemble-final` 스킬 4절과 같은 원리). `to_long`이 이미 `actual_kwh`를 포함하도록 돼 있어(위 3절 정의) 그대로 재사용한다.

In [5]:
mlp_best_epochs = []

for fold in FOLDS:
    fold_train = make_fold_train(fold)
    long_df = to_long(fold_train)
    long_sorted = long_df.sort_values("forecast_kst_dtm").reset_index(drop=True)
    n_early = int(len(long_sorted) * 0.15)
    fit_rows, early_rows = long_sorted.iloc[:-n_early], long_sorted.iloc[-n_early:]

    fit_X, fit_mu, fit_sd = mlp_nn.build_features(fit_rows, FEATURE_COLS)
    early_X, _, _ = mlp_nn.build_features(early_rows, FEATURE_COLS, mu=fit_mu, sd=fit_sd)

    fit_util = fit_rows["utilization"].to_numpy(dtype=np.float32)
    fit_kwh = fit_rows["actual_kwh"].to_numpy(dtype=np.float32)
    fit_scored = fit_util >= 0.10
    early_util = early_rows["utilization"].to_numpy(dtype=np.float32)
    early_kwh = early_rows["actual_kwh"].to_numpy(dtype=np.float32)
    early_scored = early_util >= 0.10

    _, best_val_score, best_epoch = mlp_nn.train_mlp(
        fit_X, fit_util, fit_kwh, fit_scored,
        early_X, early_util, early_kwh, early_scored,
        input_dim=fit_X.shape[1], seed=42, T_soft=MLP_T_SOFT,
        hidden=MLP_HIDDEN, dropout=MLP_DROPOUT,
        max_epochs=MLP_MAX_EPOCHS_CAP, patience=MLP_PATIENCE,
    )
    mlp_best_epochs.append(best_epoch)
    print(f"{fold['name']}: best_epoch={best_epoch} (val_score={best_val_score:.4f})")

mlp_mean_epoch = float(np.mean(mlp_best_epochs))
MLP_FINAL_EPOCHS = int(np.ceil(mlp_mean_epoch * 1.05))
print(f"\n평균 best_epoch: {mlp_mean_epoch:.1f} -> 1.05배 적용 후 MLP_FINAL_EPOCHS = {MLP_FINAL_EPOCHS}")

fold1: best_epoch=37 (val_score=0.6160)
fold2: best_epoch=56 (val_score=0.6130)
fold3: best_epoch=77 (val_score=0.6333)

평균 best_epoch: 56.7 -> 1.05배 적용 후 MLP_FINAL_EPOCHS = 60


## 4. 전체 데이터 재학습 (seed 앙상블)

`ensemble-final` 스킬 4절대로 **train 전체(2022~2024)**를 그대로 학습에 쓴다(더 이상 fold로 나누지 않음 — 검증이 아니라 최종 제출용 모델이므로). `ENSEMBLE_SEEDS` 각각으로 독립 학습해 `models/`에 저장한다. `iterations`는 3절에서 정한 `FINAL_ITERATIONS`로 고정하고 early stopping은 쓰지 않는다(검증셋이 없으므로).

In [6]:
full_long = to_long(train_features)
full_long["group_id"] = pd.Categorical(full_long["group_id"], categories=GROUP_ID_CATEGORIES)

fit_kwargs_full = {"sample_weight": full_long["actual_kwh"].to_numpy()} if USE_SAMPLE_WEIGHT else {}

trained_models = {}
for seed in ENSEMBLE_SEEDS:
    model = CatBoostRegressor(
        loss_function=f"Quantile:alpha={QUANTILE_ALPHA}",
        iterations=FINAL_ITERATIONS,
        random_seed=seed,
        verbose=False,
        **FINAL_HYPERPARAMS,
    )
    model.fit(
        full_long[features], full_long["utilization"],
        cat_features=["group_id"], verbose=False,
        **fit_kwargs_full,
    )
    model_path = MODELS_DIR / f"catboost_seed{seed}.cbm"
    model.save_model(str(model_path))
    trained_models[seed] = model
    print(f"seed={seed} 학습 완료, 저장: {model_path}")

seed=42 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed42.cbm
seed=7 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed7.cbm
seed=2024 학습 완료, 저장: d:\공모전\wind_forecast\models\catboost_seed2024.cbm


## 4b. MLP 전체 데이터 재학습 (seed 앙상블)

CatBoost와 마찬가지로 train 전체(2022~2024)로 학습한다. 표준화 통계(`mu`/`sd`)는 전체 학습 데이터로 한 번만 fit하고(모든 seed가 공유), 각 seed는 가중치 초기화만 다르게 해서 `MLP_FINAL_EPOCHS` 고정 epoch만큼 학습한다(`src.nn.train_mlp_full` — early stopping 없음).

In [7]:
mlp_full_X, mlp_mu, mlp_sd = mlp_nn.build_features(full_long, FEATURE_COLS)
mlp_full_util = full_long["utilization"].to_numpy(dtype=np.float32)
mlp_full_kwh = full_long["actual_kwh"].to_numpy(dtype=np.float32)
mlp_full_scored = mlp_full_util >= 0.10
mlp_input_dim = mlp_full_X.shape[1]

trained_mlp_models = {}
for seed in ENSEMBLE_SEEDS:
    mlp_model = mlp_nn.train_mlp_full(
        mlp_full_X, mlp_full_util, mlp_full_kwh, mlp_full_scored,
        input_dim=mlp_input_dim, epochs=MLP_FINAL_EPOCHS, seed=seed,
        T_soft=MLP_T_SOFT, hidden=MLP_HIDDEN, dropout=MLP_DROPOUT,
    )
    mlp_model_path = MODELS_DIR / f"mlp_seed{seed}.pt"
    torch.save(mlp_model.state_dict(), mlp_model_path)
    trained_mlp_models[seed] = mlp_model
    print(f"MLP seed={seed} 학습 완료, 저장: {mlp_model_path}")

scaler_path = MODELS_DIR / "mlp_scaler.npz"
np.savez(scaler_path, mu=mlp_mu, sd=mlp_sd)
print(f"표준화 통계 저장: {scaler_path}")

MLP seed=42 학습 완료, 저장: d:\공모전\wind_forecast\models\mlp_seed42.pt
MLP seed=7 학습 완료, 저장: d:\공모전\wind_forecast\models\mlp_seed7.pt
MLP seed=2024 학습 완료, 저장: d:\공모전\wind_forecast\models\mlp_seed2024.pt
표준화 통계 저장: d:\공모전\wind_forecast\models\mlp_scaler.npz


## 5. 재현용 설정 저장

`inference.ipynb`가 이 노트북과 독립적으로 실행되면서도 같은 설정(피처 목록, 시드, 손실 함수, 그룹 매핑 등)을 그대로 쓸 수 있도록 `models/model_config.json`에 저장한다.

In [8]:
model_config = {
    "feature_cols": FEATURE_COLS,
    "quantile_alpha": QUANTILE_ALPHA,
    "use_sample_weight": USE_SAMPLE_WEIGHT,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "final_iterations": FINAL_ITERATIONS,
    "hyperparams": FINAL_HYPERPARAMS,
    "group_id_map": GROUP_ID_MAP,
    "capacity_kwh": CAPACITY_KWH,
    "catboost_version": catboost.__version__,
    "mlp": {
        "t_soft": MLP_T_SOFT,
        "hidden": list(MLP_HIDDEN),
        "dropout": MLP_DROPOUT,
        "final_epochs": MLP_FINAL_EPOCHS,
        "input_dim": mlp_input_dim,
        "torch_version": torch.__version__,
    },
    "blend_weights": BLEND_WEIGHTS,
}

config_path = MODELS_DIR / "model_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(model_config, f, ensure_ascii=False, indent=2)

print(f"저장 완료: {config_path}")
model_config

저장 완료: d:\공모전\wind_forecast\models\model_config.json


{'feature_cols': ['gfs_ws10m',
  'gfs_ws80m',
  'gfs_ws100m',
  'gfs_wd100m_sin',
  'gfs_wd100m_cos',
  'gfs_ws100m_sq',
  'gfs_ws100m_cube',
  'gfs_ws850hpa',
  'kpx_group_1_ldaps_ws10m',
  'kpx_group_1_ldaps_ws10m_sq',
  'kpx_group_1_ldaps_ws10m_cube',
  'kpx_group_1_ldaps_wd10m_sin',
  'kpx_group_1_ldaps_wd10m_cos',
  'kpx_group_2_ldaps_ws10m',
  'kpx_group_2_ldaps_ws10m_sq',
  'kpx_group_2_ldaps_ws10m_cube',
  'kpx_group_2_ldaps_wd10m_sin',
  'kpx_group_2_ldaps_wd10m_cos',
  'kpx_group_3_ldaps_ws10m',
  'kpx_group_3_ldaps_ws10m_sq',
  'kpx_group_3_ldaps_ws10m_cube',
  'kpx_group_3_ldaps_wd10m_sin',
  'kpx_group_3_ldaps_wd10m_cos',
  'gfs_rho',
  'gfs_ws100m_rho_corrected',
  'kpx_group_1_ldaps_rho',
  'kpx_group_1_ldaps_ws10m_rho_corrected',
  'kpx_group_2_ldaps_rho',
  'kpx_group_2_ldaps_ws10m_rho_corrected',
  'kpx_group_3_ldaps_rho',
  'kpx_group_3_ldaps_ws10m_rho_corrected',
  'gfs_temp_diff_2m_850hpa',
  'kpx_group_1_ldaps_blh',
  'kpx_group_1_ldaps_gust_range_50m',
  'kpx_gro

## 6. Sanity check — 재학습 모델의 예측이 그럴듯한가

재학습 모델은 검증셋이 없어 Score를 낼 수 없다(`ensemble-final` 스킬 4절). 대신 **train 데이터 자체에 대한 예측(in-sample)이 실제 분포와 형태가 비슷한지**로 명백한 실수(피처 순서 오류, 그룹 매핑 오류, 블렌드 가중치 오적용 등)만 걸러낸다. 이건 성능 검증이 아니라 "완전히 잘못되지는 않았는지" 확인하는 최소한의 안전장치라는 점에 유의. CatBoost 단독 예측과 최종 블렌드 예측을 나란히 비교한다.

In [9]:
sanity_pred_cat = {}
sanity_pred_blend = {}
for g in TARGET_COLS:
    g_rows = train_features.copy()
    g_rows["group_id"] = pd.Categorical([GROUP_ID_MAP[g]] * len(g_rows), categories=GROUP_ID_CATEGORIES)

    cat_seed_preds = [
        trained_models[seed].predict(g_rows[features]) * CAPACITY_KWH[g]
        for seed in ENSEMBLE_SEEDS
    ]
    cat_pred_g = np.mean(cat_seed_preds, axis=0)

    mlp_rows = train_features.copy()
    mlp_rows["group_id"] = GROUP_ID_MAP[g]
    mlp_X_g, _, _ = mlp_nn.build_features(mlp_rows, FEATURE_COLS, mu=mlp_mu, sd=mlp_sd)
    mlp_seed_preds = [
        mlp_nn.predict_mlp(trained_mlp_models[seed], mlp_X_g) * CAPACITY_KWH[g]
        for seed in ENSEMBLE_SEEDS
    ]
    mlp_pred_g = np.mean(mlp_seed_preds, axis=0)

    w = BLEND_WEIGHTS[g]
    blend_pred_g = (1 - w) * cat_pred_g + w * mlp_pred_g

    sanity_pred_cat[g] = np.clip(cat_pred_g, 0, CAPACITY_KWH[g])
    sanity_pred_blend[g] = np.clip(blend_pred_g, 0, CAPACITY_KWH[g])

sanity_df = train_features[["forecast_kst_dtm"]].copy()
for g in TARGET_COLS:
    sanity_df[f"{g}_actual"] = train_features[g]
    sanity_df[f"{g}_cat_pred"] = sanity_pred_cat[g]
    sanity_df[f"{g}_blend_pred"] = sanity_pred_blend[g]

sanity_df["month"] = sanity_df["forecast_kst_dtm"].dt.month
monthly_cols = (
    [f"{g}_actual" for g in TARGET_COLS]
    + [f"{g}_cat_pred" for g in TARGET_COLS]
    + [f"{g}_blend_pred" for g in TARGET_COLS]
)
monthly_compare = sanity_df.groupby("month")[monthly_cols].mean()
print("월별 평균 발전량(kWh) — 실제 vs CatBoost 단독 vs 최종 블렌드(in-sample, seed 평균):")
print(monthly_compare)

for g in TARGET_COLS:
    at_zero = (sanity_pred_blend[g] <= 1e-6).mean()
    at_capacity = (sanity_pred_blend[g] >= CAPACITY_KWH[g] - 1e-6).mean()
    print(f"{g} (블렌드): 0kWh 근접 비율={at_zero:.1%}, 설비용량 근접 비율={at_capacity:.1%}")

월별 평균 발전량(kWh) — 실제 vs CatBoost 단독 vs 최종 블렌드(in-sample, seed 평균):
       kpx_group_1_actual  kpx_group_2_actual  kpx_group_3_actual  \
month                                                               
1             9355.272067         9685.125389         8498.592399   
2             6844.852532         6980.224835         4592.887846   
3             7098.556801         7681.636097         6601.090220   
4             6908.929998         7510.088488         5483.716352   
5             7052.539037         7259.455513         5449.645955   
6             4931.340902         5314.715026         3593.964154   
7             5637.695787         6207.910215         7521.339821   
8             3975.495720         4757.938298         2489.548800   
9             3184.124461         3612.100022         2046.757793   
10            4830.970539         5040.188527         3219.811920   
11            8142.289175         8170.425380         7603.051088   
12           11339.549944        1249

**해석 — 예측이 실제보다 높은 건 버그가 아니라 의도한 설계다**: 위 표를 보면 `_cat_pred`/`_blend_pred` 모두 `_actual`보다 대체로 높게 나온다. 이는 CatBoost의 분위수 회귀(τ=0.70, "중앙값이 아니라 70분위수를 예측")와 MLP의 `soft_metric_loss`(FICR 항이 실측값을 큰 쪽으로 가중하는 구조)가 둘 다 같은 방향(과소예측보다 과대예측이 유리한 채점 구조)으로 설계됐기 때문이다 — 모든 그룹·모든 월에서 비슷한 방향으로 나온다는 점(특정 월만 튀지 않음)이 "설계대로 작동한다"는 근거다.

**블렌드 예측이 CatBoost 단독과 크게 다르지 않은 범위에서 소폭 조정되는지 확인**: group_3은 블렌드 가중치가 1.0(MLP만)이라 `_cat_pred`와 `_blend_pred`가 그 그룹만 뚜렷이 달라야 정상이다 — group_1·2는 절반 안팎 섞이므로 완만한 차이여야 한다. 위 표에서 이 패턴이 그대로 보이는지가 "블렌드 가중치가 그룹별로 제대로 적용됐는지"의 실질적 확인이다.

**클리핑 비율도 정상 범위인지 확인**: 극단값(0 또는 설비용량 근접)에 몰린 예측이 비정상적으로 많지 않은지 본다 — 많다면 피처 순서 오류나 그룹 매핑 오류 같은 명백한 버그의 신호다.

**fold별 조기 종료 시점 참고**: CatBoost `best_iteration_`(fold1=356/fold2=518/fold3=1170, 평균×1.05=716)과 MLP `best_epoch`은 fold마다 차이가 있을 수 있다 — 위 3b절 출력을 참고. fold간 편차가 크면 `FINAL_ITERATIONS`/`MLP_FINAL_EPOCHS`를 좀 더 크게(가장 늦게 수렴한 fold에 가깝게) 잡는 대안도 검토 가능하다.

## 요약

- `05_tuning`에서 확정한 최종 구조 — **CatBoost(통합모델, τ=0.70, actual 가중) + MLP(`soft_metric_loss`, T_soft=0.003)를 그룹별 가중치로 블렌드**(group_1=0.4/group_2=0.5/group_3=1.0) — 로 **train 전체(2022~2024)**를 재학습했다
- 두 모델 모두 early stopping이 불가능해진 문제를 3-fold CV의 조기 종료 시점 평균×1.05로 고정값(iterations/epochs)을 정하는 방식으로 해결했다(`ensemble-final` 스킬 4절)
- 각각 seed 3개(42/7/2024) 앙상블로 학습해 `models/catboost_seed{seed}.cbm`, `models/mlp_seed{seed}.pt`, `models/mlp_scaler.npz`에 저장했다
- 재현에 필요한 모든 설정(피처 목록, CatBoost/MLP 하이퍼파라미터, 블렌드 가중치)을 `models/model_config.json`에 저장해 `inference.ipynb`가 독립적으로 실행 가능하게 했다
- Sanity check로 CatBoost 단독과 최종 블렌드 예측을 나란히 비교해 그룹별 블렌드 가중치가 의도대로 적용됐는지 확인했다(성능 검증 아님 — 명백한 오류만 거르는 용도)

**다음**: `inference.ipynb`에서 이 모델들과 `test_features_v2.parquet`으로 예측(CatBoost+MLP 블렌드) → 후처리 → `validate_submission()` → `submissions/`에 제출 파일 저장